# 📈 24-信号处理 · 01 DFT 与 FFT 与频谱分析

> 关键词：DFT、FFT、蝶形、频谱、Parseval
> 时域信号怎么看出"频率成分"？本章从 4 点 DFT 手算开始，一路到 FFT 蝶形与频谱泄漏。

## 本节要解决的问题

| # | 问题 | 对应内容 |
| --- | --- | --- |
| 1 | DFT 的公式到底是什么？ | §1 核心概念 |
| 2 | 4 点 DFT 怎么逐项手算？ | §2 手算 ✍️ |
| 3 | FFT 是怎么把 O(N²) 变成 O(NlogN) 的？ | §3 蝶形分解 |
| 4 | 为什么会有频谱泄漏？ | §4 代码实战 |

## 故事引入

一段 1 秒的语音里藏着 100Hz 的嗡嗡声和 3000Hz 的说话声——耳朵能分辨，数字怎么分辨？**DFT 把时间序列拆成不同频率正弦的加权和**，每个 $X[k]$ 就是这个频率的"含金量"。

为什么要做 DFT？因为同一段信号在**时域**和**频域**看到的东西完全不同。下图把一段心电信号同时画成时间波形（上）和频谱（下）：时域里纠缠在一起的基线漂移（BW）、肌电伪迹（EMG）与 50Hz 工频干扰（PLI），在频域里各自落在不同的频率位置——这正是"变频域做分析"最直观的动机。

![为什么要做 DFT？因为同一段信号在**时域**和**频域*](../figures/ecg_noise_tf.jpg)

> **怎么读**：上半部分时域波形看似一团乱麻；下半部分频谱把噪声成分按频率分开：低频的基线漂移、中频的肌电、50Hz 处的尖峰工频干扰，一目了然。DFT 的价值就是把"混在一起"的信号拆成"分得开"的频率成分。
>
> 📚 **图源**：Jia Y et al. (2024) *Preprocessing and Denoising Techniques for ECG and MCG*, Bioengineering 11(11):1109（开放获取，CC-BY 4.0）图 3。


## 核心概念：DFT 定义

N 点离散傅里叶变换（DFT）：

$$X[k]=\sum_{n=0}^{N-1}x[n]\,e^{-j\frac{2\pi}{N}kn},\quad k=0,1,\dots,N-1$$

记旋转因子 $W_N=e^{-j2\pi/N}$，则 $X[k]=\sum_n x[n]W_N^{kn}$。

- $k$ 对应数字频率 $\omega_k=2\pi k/N$（Hz 表示：$f_k=k\cdot f_s/N$）
- 反变换 IDTFT/IDFT 只差归一化 $1/N$ 和共轭
- **Parseval 定理**：$$\sum_{n=0}^{N-1}|x[n]|^2=\frac{1}{N}\sum_{k=0}^{N-1}|X[k]|^2$$ 时域能量 = 频域能量（按 1/N）。

## ✍️ 手算 4 点 DFT：x=[1,2,3,4]

**设定**：$N=4$，$W_4=e^{-j2\pi/4}=-j$。

**k=0**（直流分量）：$X[0]=1+2+3+4=\mathbf{10}$

**k=1**：

$$X[1]=1+2(-j)+3(-j)^2+4(-j)^3=1-2j-3+4j=\mathbf{-2+2j}$$

**k=2**：$X[2]=1+2(-1)+3(1)+4(-1)=\mathbf{-2}$

**k=3**：$X[3]=1+2j-3-4j=\mathbf{-2-2j}$（$=X[1]^*$，共轭对称）

**结果**：$X=[10,\ -2+2j,\ -2,\ -2-2j]$

**Parseval 校验**：$\sum|X|^2=|10|^2+|{-2+2j}|^2+|{-2}|^2+|{-2-2j}|^2=100+8+4+8=120$，而 $N\sum x^2=4(1+4+9+16)=120$ ✓

**手算收获**：

- 实序列 DFT 的**共轭对称性**：$X[N-k]=X[k]^*$——所以频谱只需画一半（k=0..N/2）
- 直流 $X[0]$ = 求和；$k=N/2$（Nyquist 频点）对实数输入必为实数
- **Parseval 是万能验算工具**：手算完一个 DFT，先核对能量守恒
- 直接 DFT 每点 $O(N)$、共 $O(N^2)$：100 万点就是 $10^{12}$ 次乘——必须 FFT

In [1]:
# ---- §2.1 验证：numpy 复算 4 点 DFT ----
import numpy as np
x = np.array([1.0, 2.0, 3.0, 4.0])
X = np.fft.fft(x)
print('DFT =', np.round(X, 4).tolist())
assert np.allclose(X, [10+0j, -2+2j, -2+0j, -2-2j])
powX = np.abs(X)**2
assert abs(np.sum(powX) - 4 * np.sum(x**2)) < 1e-9   # Parseval
print('Parseval: Σ|X|² =', sum(powX), '= N·Σx² =', 4*np.sum(x**2))
print('\n✓ 与手算一致')


DFT = [(10+0j), (-2+2j), (-2+0j), (-2-2j)]
Parseval: Σ|X|² = 120.0 = N·Σx² = 120.0

✓ 与手算一致


## 深入原理：FFT 蝶形分解

FFT 核心 = **分治**：把 N 点 DFT 按奇偶下标拆成两个 N/2 点 DFT。

N=4 时：偶数下标 $x_e=[1,3]$，奇数下标 $x_o=[2,4]$。

两个 2 点 DFT（蝶形）：

$$E[0]=1+3=4,\quad E[1]=1-3=-2;\qquad O[0]=2+4=6,\quad O[1]=2-4=-2$$

合成（旋转因子 $W_4^k$）：

$$X[0]=E[0]+W^0O[0]=4+6=\mathbf{10}$$

$$X[1]=E[1]+W^1O[1]=-2+(-j)(-2)=\mathbf{-2+2j}$$

$$X[2]=E[0]-W^0O[0]=4-6=\mathbf{-2}$$

$$X[3]=E[1]-W^1O[1]=-2-(-j)(-2)=\mathbf{-2-2j}$$

与直接 DFT 完全一致！

**复杂度**：每级 N/2 个蝶形、共 $\log_2 N$ 级 → $O(N\log_2N)$。N=1024 时直接法 $10^6$ vs FFT $10^4$，快 100 倍。

In [2]:
# ---- §3.1 验证：numpy 复算 FFT 蝶形分解 ----
import numpy as np
x = np.array([1.0, 2.0, 3.0, 4.0])
xe, xo = x[0::2], x[1::2]
E = np.fft.fft(xe); O = np.fft.fft(xo)
W = np.exp(-2j*np.pi/4*np.arange(2))
Xf = np.concatenate([E + W*O, E - W*O])
print('蝶形分解 =', np.round(Xf, 4).tolist())
assert np.allclose(Xf, np.fft.fft(x))
print('\n✓ 与直接 DFT 一致')


蝶形分解 = [(10+0j), (-2+2j), (-2+0j), (-2-2j)]

✓ 与直接 DFT 一致


## 代码实战：频谱与泄漏

一段 8Hz 正弦以 fs=64 采样 256 点：频率分辨率 $\Delta f=f_s/N=64/256=0.25$ Hz。若频率是分辨率整数倍（如 8Hz），峰值干净；若不是（如 8.3Hz），能量泄漏到相邻频点——这就是**频谱泄漏**，工程上用加窗缓解。

In [3]:
# ---- 频谱分析：整数倍 vs 非整数倍频率 ----
import numpy as np
fs, N = 64, 256
n = np.arange(N)
x1 = np.cos(2*np.pi*8/fs*n)      # 8Hz，整数倍分辨率 (8/0.25=32)
x2 = np.cos(2*np.pi*8.3/fs*n)    # 8.3Hz，非整数倍
def peak_energy(x):
    X = np.abs(np.fft.fft(x))**2
    k = np.argmax(X)
    return k, X[k], X[k]/X.sum()
print('8Hz   : 峰值 bin =', peak_energy(x1)[0], '，峰值占比 = %.4f' % peak_energy(x1)[2])
print('8.3Hz : 峰值 bin =', peak_energy(x2)[0], '，峰值占比 = %.4f' % peak_energy(x2)[2])
assert peak_energy(x1)[0] == 32            # 整数倍: 峰值精确落 bin 32
assert peak_energy(x2)[0] != 32            # 非整数倍: 泄漏到相邻 bin
assert peak_energy(x1)[2] > peak_energy(x2)[2]  # 泄漏使能量分散
print('✓ 整数倍频率峰值精确落 bin，非整数倍发生泄漏')


8Hz   : 峰值 bin = 32 ，峰值占比 = 0.5000
8.3Hz : 峰值 bin = 33 ，峰值占比 = 0.4382
✓ 整数倍频率峰值精确落 bin，非整数倍发生泄漏


## 面试考点

**Q1：DFT 和 DTFT 的区别？**

DTFT 是连续频率 $\omega\in[-\pi,\pi)$ 的变换；DFT 只取 N 个等间隔频点，等价于 DTFT 的抽样。

**Q2：FFT 为什么快？**

奇偶分解 + 旋转因子周期性/对称性，$O(N^2)\to O(N\log N)$；N 必须为 2 的幂（radix-2）。

**Q3：频率分辨率由什么决定？**

$\Delta f=f_s/N$：采样率固定时，**帧越长分辨率越高**（§03 STFT 权衡）。

**Q4：为什么频谱只画一半？**

实序列 DFT 共轭对称 $X[N-k]=X[k]^*$，后一半无新信息；画 $k=0..N/2$ 即可。

## 小结与下一章预告

- DFT 把时域变频域；$X=[10,-2+2j,-2,-2-2j]$ 手算 + Parseval 验算
- FFT 蝶形把 $O(N^2)$ 降到 $O(N\log N)$，手算分解与直接 DFT 一致
- 下一章：时域到频域的前提是**采样**——采样率不够会发生混叠

## ✅ 自测清单

- [ ] 能手算小 N 的 DFT（含共轭对称与 Parseval 验算）
- [ ] 能画出 4 点 FFT 蝶形图并说明复杂度
- [ ] 知道频谱泄漏的原因与加窗缓解
- [ ] 能说出 DFT/DTFT/FFT 三者关系